# Práctica: Hugging Face Transformers (Sesión NLP)

Material de práctica para el curso de **Procesamiento del Lenguaje Natural**. Aquí se usa la librería [`transformers`](https://github.com/huggingface/transformers) para cargar modelos preentrenados, emplearlos con `pipeline` y ver qué hace el **tokenizador** por debajo.

**Objetivos de esta hoja**
- Usar `pipeline` para análisis de sentimiento sin montar la arquitectura a mano.
- Cargar explícitamente `AutoTokenizer` y `AutoModelForSequenceClassification`.
- Entender la **codificación** (texto → tokens → IDs) y la **decodificación** (IDs → texto).
- Guardar el modelo en disco y ejecutar el **forward pass** en PyTorch (logits → probabilidades).

> Los textos de ejemplo son genéricos (prácticas de curso y materiales de estudio), pensados para el aula.

---

### 1. Pipeline

La API de **pipelines** agrupa tokenización, inferencia y postproceso en una sola llamada de alto nivel.

In [ ]:
# Primera celda: instala la librería (en Colab suele bastar; en local, usa el mismo entorno del curso)
!pip install transformers -q

Hugging Face mantiene la librería open source **[Transformers](https://github.com/huggingface/transformers)** y el **[Hub de modelos](https://huggingface.co/models)**. Con `pip install transformers` puedes empezar en local o en Colab; en proyectos serios conviene fijar versiones en un `requirements.txt` o en un entorno conda para que todos obtengan el mismo comportamiento.

In [ ]:
from transformers import pipeline

In [ ]:
# Tarea: clasificación de sentimiento (por defecto elige un modelo concreto del Hub)
classifier = pipeline("sentiment-analysis")

In [ ]:
res = classifier("Me parecen muy útiles estas prácticas; los ejemplos con pipeline son claros.")
print(res)

In [ ]:
res = classifier("El temario de la asignatura es amplio, pero los recursos del curso permiten avanzar con claridad.")
print(res)

#### Selección del modelo

Si no indicas `model=...`, `pipeline` usa un **modelo por defecto** (suele ser una variante en **inglés** para sentimiento). En **producción** o en entregables conviene **nombrar siempre** el modelo y, si hace falta, la **revisión** (`revision`), para que los resultados sean **reproducibles** y el idioma coincida con tus datos.

In [ ]:
# seleccionamos el mismo modelo que tenemos por defecto
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

In [ ]:
res = classifier("El temario de la asignatura es amplio, pero los recursos del curso permiten avanzar con claridad.")
print(res)

Un **pipeline** agrupa la tarea completa: tokenización, paso por el modelo y (según la tarea) formato de salida. Hay pipelines para **reconocimiento de entidades** (`"ner"`), **respuesta a preguntas**, **rellenar máscaras**, **resumen**, etc.

Documentación: [lista de pipelines y tareas](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline).

### 2. Modelo y tokenizador

`pipeline(...)` oculta varios pasos. A continuación se usan **`AutoTokenizer`** y **`AutoModelForSequenceClassification`** para cargar el mismo tipo de modelo y ver explícitamente los objetos que intervienen.

Con una sola línea de `pipeline` ya se ejecutan **varias etapas**: carga de pesos, tokenización, inferencia y mapeo de la salida a etiquetas. El código siguiente repite el flujo **mostrando** tokenizador y modelo por separado; así puedes reutilizarlos en un script propio o cambiar el postproceso.

In [ ]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# Elige un modelo del Hub por nombre. La línea comentada es un ejemplo en inglés (SST-2);
# la activa es un modelo de sentimiento en español (útil para los ejemplos de texto de esta hoja).
# model_name = "distilbert-base-uncased-finetuned-sst-2-english"
model_name = "pysentimiento/robertuito-sentiment-analysis"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

res = classifier("El temario de la asignatura es amplio, pero los recursos del curso permiten avanzar con claridad.")
print(res)

### 3. ¿Para qué sirve el tokenizador?

La red neuronal trabaja con **vectores numéricos**, no con caracteres sueltos. El **tokenizador** convierte el texto en **tokens** (subpalabras o piezas) y luego en **IDs** según el vocabulario con el que el modelo fue entrenado (WordPiece, BPE, SentencePiece, etc.).

#### Codificación (entrada al modelo)

Antes de la inferencia, el texto se **codifica**: segmentación en tokens, mapeo a IDs y, en modelos tipo BERT, inclusión de tokens especiales (`[CLS]`, `[SEP]`, etc.) y máscara de atención cuando hay varias frases o padding.

In [ ]:
secuencia = "El temario de la asignatura es amplio, pero los recursos del curso permiten avanzar con claridad."
res = tokenizer(secuencia)
print(res)

##### Paso a paso

Puedes descomponer el proceso: `tokenize` → `convert_tokens_to_ids`, o bien llamar a `tokenizer(texto)` y obtener un diccionario con `input_ids`, `attention_mask` y, si el modelo lo usa, `token_type_ids` u otros campos.

In [ ]:
tokens = tokenizer.tokenize(secuencia)
print(tokens)

In [ ]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

#### Decodificar (texto legible tras el modelo)

`decode` pasa de IDs de vuelta a texto; sirve para inspeccionar o para modelos generativos. Aquí ayuda a comprobar cómo el tokenizador **recompone** la secuencia a partir de `input_ids`.

In [ ]:
tokenizer.decode(res['input_ids'])

### 4. Guardar modelo y tokenizador en local

`save_pretrained(ruta)` escribe la configuración (`config.json`), los pesos del modelo (por ejemplo `model.safetensors` o `pytorch_model.bin`) y los archivos del tokenizador (`tokenizer_config.json`, vocabulario, etc.). Sirve para **trabajar sin conexión**, **versionar** un fine-tuning o **desplegar** el mismo artefacto en otro equipo.

In [ ]:
model_path = "./modelo"  # carpeta local; se creará si no existe
tokenizer.save_pretrained(model_path)
model.save_pretrained(model_path)

In [ ]:
tokenizer_local = AutoTokenizer.from_pretrained(model_path)
model_local = AutoModelForSequenceClassification.from_pretrained(model_path)

### 5. PyTorch: tensores, logits y probabilidades

Bajo el `pipeline`, el objeto `model` es un **`torch.nn.Module`**. Pasando tensores en lugar de strings controlas el **batch**, el **padding** y puedes leer directamente los **logits** (salidas sin normalizar antes del softmax).

La misma familia de modelos tiene API equivalente en **TensorFlow** (`TFAutoModel…`, `return_tensors="tf"`). Esta práctica usa **PyTorch** porque es el ejemplo más habitual en la documentación de Transformers y en muchos cuadernos de investigación.

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
sentences = [
    "He estado esperando practicar con modelos preentrenados y pipelines en un entorno real.",
    "Me encanta la idea del Hub de modelos y la documentación de Hugging Face.",
]

#### Tokenizar el batch

`padding=True` iguala longitudes dentro del batch; `truncation=True` recorta si se supera `max_length`; `return_tensors="pt"` devuelve tensores listos para PyTorch en lugar de listas de Python.

In [ ]:
batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)
print(batch)

#### Forward del modelo (inferencia)

`torch.no_grad()` desactiva el cálculo de gradientes (solo inferencia, menos memoria). `outputs.logits` tiene forma `[batch_size, num_labels]`; **`softmax`** convierte logits en probabilidades por fila y **`argmax`** da el índice de la clase predicha para cada frase.

In [ ]:
with torch.no_grad():
    outputs = model(**batch)
    predictions = F.softmax(outputs.logits, dim=1)
    labels = torch.argmax(predictions, dim=1)

In [ ]:
# Salida cruda del modelo: logits (un valor por clase antes del softmax)
print(outputs)

In [ ]:
# Probabilidades por clase (robertuito: neg / neu / pos según el orden del config del modelo)
print(predictions)

In [ ]:
# Índice de la clase con mayor probabilidad por cada fila del batch
print(labels)

In [ ]:
# Inspección opcional: arquitectura y cabeza de clasificación
model